# Blowing-snow cloud top across model resolutions

CRYOWRF `bs_qi` profile at PEA (bilinear interpolation) for the five nested domains:
d01 27 km, d02 9 km, d03 3 km, d04 1 km, d05 200 m. Cloud top = height where `bs_qi` drops below
10⁻⁶ kg kg⁻¹ (linear interpolation between model levels). The ceilometer cloud top from the BLSN
algorithm is the reference. `PLOT_EVENT` selects one event or `"all"`.

In [ ]:
# ═══════════════════════════════════════════════════════════════════════════
#  CONFIG
# ═══════════════════════════════════════════════════════════════════════════
import os

SIMULATIONS = {
    "warm_syn": {
        "sim_dir": r"D:\warm_syn_CRYOWRF",
        "start":   "2025-04-24 12:00",
        "end":     "2025-04-25 12:00",
        "blsn_csv": r"C:\Users\lory2\Desktop\tesi\BLSN\good_notebooks\blsn_2025_v1505_jump_25.csv",
        "label":   "Warm synoptic – Apr 2025",
    },
    "trans_syn": {
        "sim_dir": r"D:\trans_syn_CRYOWRF",
        "start":   "2025-04-23 05:00",
        "end":     "2025-04-24 00:00",
        "blsn_csv": r"C:\Users\lory2\Desktop\tesi\BLSN\good_notebooks\blsn_2025_v1505_jump_25.csv",
        "label":   "Transitional synoptic – Apr 2025",
    },
    "cold_kat": {
        "sim_dir": r"D:\cold_kat_CRYOWRF",
        "start":   "2024-05-13 01:00",
        "end":     "2024-05-14 14:00",
        "blsn_csv": r"C:\Users\lory2\Desktop\tesi\BLSN\good_notebooks\blsn_2024_v1505_jump_25.csv",
        "label":   "Cold katabatic – May 2024",
    },
}

# Domain → grid-spacing label. Order = coarse → fine (used everywhere for x-axis / legend).
DOMAIN_RES = {
    "d01": "27 km",
    "d02": "9 km",
    "d03": "3 km",
    "d04": "1 km",
    "d05": "200 m",
}

# ── What to plot ────────────────────────────────────────────────────────────
# "cold_kat" | "warm_syn" | "trans_syn"  → single event
# "all"                                  → all three, one panel per event
PLOT_EVENT = "all"

BSQI_THRESHOLD    = 1e-6   # kg kg⁻¹  — cloud top = height where bs_qi drops below this
ALIGN_WINDOW_MIN  = 10     # ceilometer averaged in ±5 min around each WRF timestamp
PEA_ROOF_HEIGHT_M = 15.0   # m — ceilometer height on the PEA roof, added to bring obs to AGL

# One colour per resolution (coarse = dark, fine = bright), assigned in the imports cell
EVENT_ORDER = ["cold_kat", "trans_syn", "warm_syn"]

OUTPUT_DIR = r"C:\Users\lory2\Desktop\tesi\wrf_notebooks\validation_plots"
DPI        = 180
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Config loaded.")
print("Events:", list(SIMULATIONS.keys()))
print("Domains:", DOMAIN_RES)
print("Threshold:", f"{BSQI_THRESHOLD:.1e} kg/kg")
print("PLOT_EVENT =", PLOT_EVENT)


In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import glob
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from scipy.interpolate import RegularGridInterpolator
from scipy.stats import pearsonr
from netCDF4 import Dataset

PEA_LAT = -71.94960
PEA_LON =  23.34988
G       = 9.80665   # m s⁻²

# Colour per resolution: coarse (dark) → fine (bright), following DOMAIN_RES order
_res_labels = list(DOMAIN_RES.values())
_cmap = plt.cm.viridis(np.linspace(0.05, 0.9, len(_res_labels)))
RES_COLORS = {lab: _cmap[i] for i, lab in enumerate(_res_labels)}

print("Imports OK")
print("Resolution colours:", _res_labels)


In [ ]:
# ── Helper functions (adapted from cloud_top_validation.ipynb) ───────────────

def _get_latlon(nc):
    lats = np.asarray(nc.variables["XLAT" ][0, :, 0], float)
    lons = np.asarray(nc.variables["XLONG"][0, 0, :], float)
    return lats, lons


def _interp_pt(lats, lons, field2d):
    """Bilinear interpolation of field2d (ny, nx) to PEA."""
    itp = RegularGridInterpolator(
        (lats, lons), field2d, method="linear",
        bounds_error=False, fill_value=np.nan
    )
    return float(itp([[PEA_LAT, PEA_LON]])[0])


def _decode_wrf_times(nc):
    raw = nc.variables["Times"][:]
    if raw.dtype.kind in ("S", "U"):
        strs = ["".join(r.astype(str)) for r in raw]
    else:
        strs = ["".join(r.tobytes().decode("ascii", errors="replace")) for r in raw]
    return [pd.Timestamp(s.replace("_", " ")) for s in strs]


def get_wrf_files(sim_dir, domain):
    files = sorted(glob.glob(f"{sim_dir}/wrfout_{domain}_*.nc"))
    if not files:
        raise FileNotFoundError(f"No wrfout files in {sim_dir}/wrfout_{domain}_*.nc")
    return files


def _mass_heights_pea(nc, ti=0):
    """Height AGL [m] at the 96 mass levels, interpolated to PEA."""
    lats, lons = _get_latlon(nc)
    ph  = np.asarray(nc.variables["PH" ][ti], float)
    phb = np.asarray(nc.variables["PHB"][ti], float)
    hgt = np.asarray(nc.variables["HGT"][ti], float)
    z_w = (ph + phb) / G - hgt[np.newaxis, :, :]
    z_m = 0.5 * (z_w[:-1] + z_w[1:])
    n = z_m.shape[0]
    return np.array([_interp_pt(lats, lons, z_m[k]) for k in range(n)])


def cloud_top_absolute(z_m, bsqi_profile, threshold):
    """Cloud-top height [m AGL] = first height where bs_qi drops below `threshold`.
    Linear interpolation between adjacent levels. NaN if surface value <= threshold."""
    if bsqi_profile[0] <= threshold or not np.isfinite(bsqi_profile[0]):
        return np.nan
    for k in range(1, len(bsqi_profile)):
        if bsqi_profile[k] <= threshold:
            q_lo, q_hi = bsqi_profile[k - 1], bsqi_profile[k]
            z_lo, z_hi = z_m[k - 1], z_m[k]
            if q_hi == q_lo:
                return z_lo
            t = (threshold - q_lo) / (q_hi - q_lo)
            return z_lo + t * (z_hi - z_lo)
    nz = np.where(np.asarray(bsqi_profile) > threshold)[0]
    return float(z_m[nz[-1]]) if len(nz) else np.nan


def extract_wrf_cloud_tops(sim_dir, domain, threshold=BSQI_THRESHOLD):
    """For each WRF timestep of `domain`: bs_qi profile at PEA → cloud-top height.
    Returns hourly pd.Series [m AGL]."""
    files = get_wrf_files(sim_dir, domain)
    times_all, tops_all = [], []
    for f in files:
        nc = Dataset(f)
        lats, lons = _get_latlon(nc)
        z_m = _mass_heights_pea(nc, ti=0)
        nz  = z_m.shape[0]
        bsqi_var = nc.variables["bs_qi"]   # (n_t, nz, ny, nx)
        ft = _decode_wrf_times(nc)
        for ti in range(bsqi_var.shape[0]):
            bsqi_3d = np.asarray(bsqi_var[ti], float)
            profile = np.array([_interp_pt(lats, lons, bsqi_3d[k]) for k in range(nz)])
            tops_all.append(cloud_top_absolute(z_m, profile, threshold))
            times_all.append(ft[ti])
        nc.close()
    idx = pd.DatetimeIndex(times_all).tz_localize(None)
    return pd.Series(tops_all, index=idx, name="wrf_cloud_top_m").sort_index()


def load_blsn_top(csv_path, t0, t1, category_keep=(1, 2, 3), roof_height=None):
    """Ceilometer cloud-top height [m AGL], 5-min, NaN for non-BLSN timesteps."""
    if roof_height is None:
        roof_height = PEA_ROOF_HEIGHT_M
    FILL = 99999999.9
    df = pd.read_csv(csv_path)
    df["datetime"] = pd.to_datetime(df["datetime"], format="mixed")
    df = df.set_index("datetime").sort_index()
    df = df[~df.index.duplicated(keep="first")]
    df.index = df.index.tz_localize(None)
    top = df["top_of_detected_BLSN"].astype(float)
    top = top.where(top < FILL * 0.5, np.nan)
    cat = df["category"].astype(int)
    top = top.where(cat.isin(category_keep), np.nan)
    top = top + roof_height
    return top.loc[t0:t1].rename("ceilo_cloud_top_m")


def align_to_wrf(obs_series, wrf_index, window_min=ALIGN_WINDOW_MIN):
    """Average obs within ± window_min/2 minutes of each WRF timestamp."""
    half = pd.Timedelta(minutes=window_min / 2)
    aligned = [obs_series.loc[t - half : t + half].mean() for t in wrf_index]
    return pd.Series(aligned, index=wrf_index, name=obs_series.name)


def metrics(obs_s, mod_s):
    """MAE, Bias (mod−obs), Pearson r over common valid points."""
    o, m = obs_s.align(mod_s, join="inner")
    o = np.asarray(o, float); m = np.asarray(m, float)
    mask = np.isfinite(o) & np.isfinite(m)
    if mask.sum() < 2:
        return dict(n=int(mask.sum()), mae=np.nan, bias=np.nan, r=np.nan)
    o, m = o[mask], m[mask]
    r, _ = pearsonr(o, m)
    return dict(n=int(mask.sum()), mae=float(np.mean(np.abs(o - m))),
                bias=float(np.mean(m - o)), r=float(r))

print("Helper functions defined.")


In [ ]:
# ── Extract cloud tops for every event × domain, plus ceilometer ─────────────
# results[event] = {
#     "label", "wrf": {res_label: Series}, "ceilo_native": Series, "ceilo_h": Series
# }
results = {}

for key, sim in SIMULATIONS.items():
    t0 = pd.Timestamp(sim["start"]); t1 = pd.Timestamp(sim["end"])
    print(f"\n{'─'*64}\n{key}   {t0} → {t1}")

    wrf_by_res = {}
    finest_index = None
    for dom, res in DOMAIN_RES.items():
        try:
            s = extract_wrf_cloud_tops(sim["sim_dir"], dom, BSQI_THRESHOLD).loc[t0:t1]
        except FileNotFoundError as e:
            print(f"  [{res:>5}] {dom}: MISSING — {e}")
            continue
        wrf_by_res[res] = s
        finest_index = s.index   # last successful (finest) domain index
        print(f"  [{res:>5}] {dom}: {len(s)} steps, "
              f"{s.notna().sum()} with model BLSN, "
              f"mean top = {np.nanmean(s.values):.0f} m" if s.notna().any()
              else f"  [{res:>5}] {dom}: {len(s)} steps, no model BLSN")

    ceilo = load_blsn_top(sim["blsn_csv"], t0, t1)
    ceilo_h = align_to_wrf(ceilo, finest_index) if finest_index is not None else pd.Series(dtype=float)

    results[key] = dict(label=sim["label"], wrf=wrf_by_res,
                        ceilo_native=ceilo, ceilo_h=ceilo_h)

print("\nExtraction done. Events ready:", list(results.keys()))


## 1. Cloud top time series by resolution

In [ ]:
# ── Time-series plot: cloud top by resolution ────────────────────────────────

def _panel_timeseries(ax, key):
    r = results[key]
    # ceilometer reference
    if r["ceilo_native"].notna().any():
        ax.plot(r["ceilo_native"].index, r["ceilo_native"].values,
                color="black", lw=0.7, alpha=0.30, label="Ceilometer 5-min")
    if len(r["ceilo_h"]) and r["ceilo_h"].notna().any():
        ax.plot(r["ceilo_h"].index, r["ceilo_h"].values,
                color="black", lw=1.8, marker="o", ms=4,
                label="Ceilometer (1-h mean)")
    # one line per resolution, coarse → fine
    for res in DOMAIN_RES.values():
        if res not in r["wrf"]:
            continue
        s = r["wrf"][res]
        ax.plot(s.index, s.values, color=RES_COLORS[res], lw=2.0,
                marker="s", ms=4, label=f"CRYOWRF {res}")
    ax.set_title(r["label"], fontsize=13)
    ax.set_ylabel("Cloud-top height [m AGL]", fontsize=12)
    ax.grid(True, alpha=0.35); ax.set_ylim(bottom=0)
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%d %b\n%H:%M"))
    ax.legend(fontsize=8, loc="upper right", ncol=2)


def plot_timeseries(which=None):
    which = which if which is not None else PLOT_EVENT
    keys = EVENT_ORDER if which == "all" else [which]
    keys = [k for k in keys if k in results]
    fig, axes = plt.subplots(len(keys), 1, figsize=(11, 3.4 * len(keys)), squeeze=False)
    for ax, key in zip(axes[:, 0], keys):
        _panel_timeseries(ax, key)
    ttl = "all events" if which == "all" else results[keys[0]]["label"]
    fig.suptitle(f"Blowing-snow cloud top by resolution  [bs_qi < {BSQI_THRESHOLD:.0e} kg/kg]  —  {ttl}",
                 fontsize=15, fontweight="bold", y=1.005)
    plt.tight_layout(); plt.show()
    tag = "all" if which == "all" else which
    fig.savefig(os.path.join(OUTPUT_DIR, f"cloud_top_by_resolution_{tag}.png"),
                dpi=DPI, bbox_inches="tight")
    print("Saved:", f"cloud_top_by_resolution_{tag}.png")


plot_timeseries()   # uses PLOT_EVENT; or call plot_timeseries("cold_kat")


## 2. Mean cloud top vs resolution

Bars: mean; whiskers: min–max over the event; dashed line: mean ceilometer cloud top.

In [ ]:
# ── Summary bars: mean cloud top per resolution ──────────────────────────────

def _res_stats(series):
    v = np.asarray(series.values, float); v = v[np.isfinite(v)]
    if v.size == 0:
        return np.nan, np.nan, np.nan
    return float(np.mean(v)), float(np.min(v)), float(np.max(v))


def plot_summary(which=None):
    which = which if which is not None else PLOT_EVENT
    keys = EVENT_ORDER if which == "all" else [which]
    keys = [k for k in keys if k in results]
    res_labels = list(DOMAIN_RES.values())
    x = np.arange(len(res_labels))

    fig, axes = plt.subplots(1, len(keys), figsize=(4.6 * len(keys), 4.6),
                             squeeze=False, sharey=True)
    for ax, key in zip(axes[0], keys):
        r = results[key]
        means, los, his = [], [], []
        for res in res_labels:
            if res in r["wrf"]:
                mu, lo, hi = _res_stats(r["wrf"][res])
            else:
                mu = lo = hi = np.nan
            means.append(mu); los.append(lo); his.append(hi)
        means = np.array(means); los = np.array(los); his = np.array(his)
        yerr = np.vstack([means - los, his - means])
        yerr = np.where(np.isfinite(yerr), yerr, 0.0)
        colors = [RES_COLORS[res] for res in res_labels]
        ax.bar(x, means, yerr=yerr, capsize=4, color=colors, alpha=0.85,
               error_kw=dict(ecolor="#444444", lw=1))
        # ceilometer reference
        if len(r["ceilo_h"]) and r["ceilo_h"].notna().any():
            cmean = float(np.nanmean(r["ceilo_h"].values))
            ax.axhline(cmean, color="black", ls="--", lw=1.4,
                       label=f"Ceilometer mean ({cmean:.0f} m)")
            ax.legend(fontsize=9, loc="upper right")
        ax.set_xticks(x); ax.set_xticklabels(res_labels)
        ax.set_title(r["label"], fontsize=12)
        ax.set_xlabel("Grid resolution (coarse → fine)", fontsize=11)
        ax.grid(axis="y", alpha=0.35); ax.set_ylim(bottom=0)
    axes[0][0].set_ylabel("Mean cloud-top height [m AGL]  (whiskers = min–max)", fontsize=11)

    ttl = "all events" if which == "all" else results[keys[0]]["label"]
    fig.suptitle(f"Mean blowing-snow cloud top vs resolution  —  {ttl}",
                 fontsize=14, fontweight="bold", y=1.02)
    plt.tight_layout(); plt.show()
    tag = "all" if which == "all" else which
    fig.savefig(os.path.join(OUTPUT_DIR, f"cloud_top_summary_{tag}.png"),
                dpi=DPI, bbox_inches="tight")
    print("Saved:", f"cloud_top_summary_{tag}.png")


plot_summary()   # uses PLOT_EVENT; or call plot_summary("warm_syn")


## 3. Statistics

Per event and resolution: model cloud-top statistics and MAE, bias (model − obs) and Pearson r against the ceilometer.

In [ ]:
# ── Statistics table ─────────────────────────────────────────────────────────

def stats_table(which=None):
    which = which if which is not None else PLOT_EVENT
    keys = EVENT_ORDER if which == "all" else [which]
    keys = [k for k in keys if k in results]
    rows = []
    for key in keys:
        r = results[key]
        for res in DOMAIN_RES.values():
            if res not in r["wrf"]:
                continue
            s = r["wrf"][res]
            mu, lo, hi = _res_stats(s)
            m = metrics(r["ceilo_h"], s) if len(r["ceilo_h"]) else dict(n=0, mae=np.nan, bias=np.nan, r=np.nan)
            rows.append({
                "Event": r["label"], "Resolution": res,
                "N (model BLSN)": int(np.isfinite(s.values).sum()),
                "Mean top [m]": round(mu, 1) if np.isfinite(mu) else "–",
                "Min [m]": round(lo, 1) if np.isfinite(lo) else "–",
                "Max [m]": round(hi, 1) if np.isfinite(hi) else "–",
                "N vs ceilo": m["n"],
                "MAE [m]": round(m["mae"], 1) if np.isfinite(m["mae"]) else "–",
                "Bias [m] (mod−obs)": round(m["bias"], 1) if np.isfinite(m["bias"]) else "–",
                "Pearson r": round(m["r"], 3) if np.isfinite(m["r"]) else "–",
            })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    return df

stats_table()


## 4. Time–height sections

Ceilometer backscatter minus clear sky with the detected BLSN top, and `bs_qi` at each resolution with the model cloud top. Set `TH_EVENT` and `TH_YMAX`.

In [ ]:
# ── Time–height: ceilometer signal−clear-sky + bs_qi per resolution ──────────
# Both panels use the SAME representation as the existing ceilometer plots:
# contourf + LogNorm + Blues (smooth), NOT pcolormesh (which draws the hourly
# model grid as blocky rectangles).
import warnings
from pathlib import Path
from matplotlib.colors import LogNorm
import matplotlib.ticker as mticker
from matplotlib.lines import Line2D
try:
    import xarray as xr
    _HAS_XR = True
except ImportError:
    _HAS_XR = False

# ── choose the event and the height axis top ────────────────────────────────
TH_EVENT = "trans_syn"   # "cold_kat" | "warm_syn" | "trans_syn"
TH_YMAX  = 2000         # m — top of the height axis (as in the existing plots)
TH_VMIN  = 1e-9         # kg/kg — bs_qi colour-scale floor
EXTRA_CONTOURS = [1e-8]  # extra bs_qi iso-lines [kg/kg]; set to [] to hide them
CMAP     = "Blues"      # same colormap as the existing ceilometer plots

# ── ceilometer inputs (raw backscatter .nc + clear-sky profile) ─────────────
CEILO_PATHS = {
    2024: r"D:/data/ceilo/2024",
    2025: r"D:/data/ceilo/2025",
}
CLEAR_SKY_FILE = r"C:\Users\lory2\Desktop\tesi\BLSN\good_notebooks\clear_sky_profile_bin1_21.csv"


def load_ceilo_signal(event, t0, t1):
    """Return (time_index, height [m], signal_minus_clearsky [n_t, n_h]) for the
    ceilometer over the event window, or None if data are unavailable."""
    if not _HAS_XR:
        print("  ceilometer: xarray not available — panel skipped"); return None
    ceil_path = CEILO_PATHS.get(t0.year)
    if ceil_path is None:
        print(f"  ceilometer: no path for year {t0.year} — panel skipped"); return None
    try:
        clear_sky = pd.read_csv(CLEAR_SKY_FILE, header=None).values.flatten()
    except Exception as e:
        print(f"  ceilometer: clear-sky file not read ({e}) — panel skipped"); return None

    dates = pd.date_range(t0.normalize(), t1.normalize(), freq="D").strftime("%Y%m%d")
    profs, times, height = [], [], None
    for date_str in dates:
        yyyymm = date_str[:6]
        matches = sorted(Path(ceil_path).joinpath(yyyymm).glob(f"peaceilCL31.b1.{date_str}*.nc"))
        if not matches:
            continue
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                ds = xr.open_dataset(matches[0])
            time_dt = pd.to_datetime(date_str) + pd.to_timedelta(ds["time"].values, unit="h")
            bs = ds["backscatter"]
            if bs.dims[0] != "time":
                bs = bs.transpose("time", "range")
            bs = bs.assign_coords(time=time_dt)
            bs_res = bs.resample(time="5min").mean()
            profs.append(bs_res.values)
            times.append(pd.to_datetime(bs_res["time"].values))
            if height is None:
                height = ds["range"].values
            ds.close()
        except Exception as e:
            print(f"  ceilometer {date_str}: error {e}")
    if not profs:
        print("  ceilometer: no .nc files found in window — panel skipped"); return None

    B   = np.concatenate(profs, axis=0)          # (n_t, n_h)
    idx = pd.DatetimeIndex(np.concatenate(times))
    order = np.argsort(idx.values); idx, B = idx[order], B[order]
    keep = (idx >= t0) & (idx <= t1)
    idx, B = idx[keep], B[keep]
    n = min(B.shape[1], len(clear_sky), len(height))
    signal = B[:, :n] - clear_sky[None, :n]      # signal − clear sky
    return idx, np.asarray(height[:n], float), signal


def _raw_algo_top(event, t0, t1):
    """Raw BLSN algorithm top (above instrument, no roof offset) for the ceilo panel."""
    sim = SIMULATIONS[event]
    df = pd.read_csv(sim["blsn_csv"])
    df["datetime"] = pd.to_datetime(df["datetime"], format="mixed")
    df = df.set_index("datetime").sort_index()
    df.index = df.index.tz_localize(None)
    top = df["top_of_detected_BLSN"].astype(float)
    top = top.where(top < 99999999.9 * 0.5, np.nan)
    top = top.where(df["category"].astype(int).isin([1, 2, 3]), np.nan)
    return top.loc[t0:t1]


def extract_wrf_bsqi_curtain(sim_dir, domain, t0, t1):
    """Return (time_index, z_m [nz], M [n_t, nz]) of bs_qi at PEA for a domain."""
    files = get_wrf_files(sim_dir, domain)
    times_all, rows, z_ref = [], [], None
    for f in files:
        nc = Dataset(f)
        lats, lons = _get_latlon(nc)
        z_m = _mass_heights_pea(nc, ti=0)
        if z_ref is None:
            z_ref = z_m
        nz = z_m.shape[0]
        bsqi_var = nc.variables["bs_qi"]
        ft = _decode_wrf_times(nc)
        for ti in range(bsqi_var.shape[0]):
            bsqi_3d = np.asarray(bsqi_var[ti], float)
            rows.append(np.array([_interp_pt(lats, lons, bsqi_3d[k]) for k in range(nz)]))
            times_all.append(ft[ti])
        nc.close()
    idx = pd.DatetimeIndex(times_all).tz_localize(None)
    M = np.array(rows)                       # (n_t, nz)
    order = np.argsort(idx.values)
    idx, M = idx[order], M[order]
    keep = (idx >= t0) & (idx <= t1)
    return idx[keep], z_ref, M[keep]


def _decade_colorbar(cbar, vmin, vmax):
    """Label a log colorbar only at powers of ten (10^n), no decimal clutter."""
    lo = int(np.floor(np.log10(vmin))); hi = int(np.ceil(np.log10(vmax)))
    ticks = 10.0 ** np.arange(lo, hi + 1)
    ticks = ticks[(ticks >= vmin) & (ticks <= vmax)]
    if len(ticks):
        cbar.set_ticks(ticks)
    cbar.ax.yaxis.set_major_formatter(mticker.LogFormatterMathtext())


def _log_levels(vmin, vmax, n=50):
    vmin = max(vmin, 1e-30)
    if not (vmax > vmin):
        vmax = vmin * 10
    return np.logspace(np.log10(vmin), np.log10(vmax), n)


def _fmt_time_axis(ax):
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d\n%H:%M"))
    ax.xaxis.set_major_locator(mdates.HourLocator(interval=6))
    plt.setp(ax.xaxis.get_majorticklabels(), rotation=0, ha="center")


# marker on the hourly model samples? (honest about the hourly output cadence)
SHOW_HOURLY_MARKERS = True


import math

def plot_curtains(event=None):
    event = event if event is not None else TH_EVENT
    if event == "all":
        raise ValueError("Time-height curtains are for a single event; set TH_EVENT.")
    sim = SIMULATIONS[event]
    t0, t1 = pd.Timestamp(sim["start"]), pd.Timestamp(sim["end"])
    label = results.get(event, {}).get("label", sim["label"])

    ceilo = load_ceilo_signal(event, t0, t1)

    curtains = {}
    for dom, res in DOMAIN_RES.items():
        try:
            idx, z_m, M = extract_wrf_bsqi_curtain(sim["sim_dir"], dom, t0, t1)
        except FileNotFoundError as e:
            print(f"  [{res:>5}] {dom}: MISSING — {e}"); continue
        curtains[res] = (idx, z_m, M)
    if not curtains:
        print("No model domains available for", event); return

    # shared bs_qi colour scale across resolutions
    qvmax = max((np.nanmax(M[np.isfinite(M) & (M > 0)]) if (np.isfinite(M) & (M > 0)).any() else TH_VMIN)
                for (_, _, M) in curtains.values())
    qlevels = _log_levels(TH_VMIN, qvmax, 50)
    qnorm   = LogNorm(vmin=TH_VMIN, vmax=max(qvmax, TH_VMIN * 10))

    # ── 3 rows × 2 columns grid ──
    n_panels = len(curtains) + (1 if ceilo is not None else 0)
    ncols = 2
    nrows = math.ceil(n_panels / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(16, 3 * nrows),
                             squeeze=False, sharex=True, constrained_layout=True)
    axl = axes.flatten()          # row-major: [ (0,0), (0,1), (1,0), (1,1), ... ]
    ai = 0

    # ── ceilometer panel (first): signal − clear sky, contourf + LogNorm ──
    if ceilo is not None:
        c_idx, c_h, c_sig = ceilo
        Xc = mdates.date2num(c_idx.to_pydatetime())
        pos = c_sig[np.isfinite(c_sig) & (c_sig > 0)]
        cvmin = np.nanpercentile(pos, 5)    if pos.size else 1.0
        cvmax = np.nanpercentile(pos, 99.5) if pos.size else 10.0
        clevels = _log_levels(cvmin, cvmax, 50)
        cnorm   = LogNorm(vmin=cvmin, vmax=max(cvmax, cvmin * 10))
        Cz = np.where(np.isfinite(c_sig) & (c_sig > 0), c_sig, np.nan)  # excess only

        ax = axl[ai]; ai += 1
        cf = ax.contourf(Xc, c_h, Cz.T, levels=clevels, norm=cnorm,
                         cmap=CMAP, extend="both")
        algo_top = _raw_algo_top(event, t0, t1)
        if algo_top.notna().any():
            ax.plot(mdates.date2num(algo_top.index.to_pydatetime()), algo_top.values,
                    color="black", lw=2.5, label="Detected blowing snow top", zorder=5)
            ax.legend(fontsize=11, loc="upper right")
        ax.set_ylim(0, TH_YMAX); ax.set_ylabel("Height AGL (m)", fontsize=12)
        ax.tick_params(axis="both", labelsize=12)
        ax.set_title("Ceilometer backscatter − clear sky signal",
                     fontsize=13, fontweight="bold")
        cbar_c = fig.colorbar(cf, ax=ax, pad=0.01, shrink=0.85, aspect=18,
                             label="Backscatter − clear sky (sr⁻¹ m⁻¹)")
        _decade_colorbar(cbar_c, cvmin, cvmax)

    # ── bs_qi panels: SAME contourf + LogNorm representation ──
    _extra_colors = {1e-5: "#ff7f0e", 1e-4: "#d62728"}
    q_axes, cf_q = [], None
    for res, (idx, z_m, M) in curtains.items():
        ax = axl[ai]; ai += 1; q_axes.append(ax)
        X  = mdates.date2num(idx.to_pydatetime())
        Mz = np.where(np.isfinite(M) & (M > 0), M, np.nan)
        cf_q = ax.contourf(X, z_m, Mz.T, levels=qlevels, norm=qnorm,
                           cmap=CMAP, extend="both")
        handles = []
        # model cloud top (height where bs_qi crosses the 1e-6 threshold)
        if event in results and res in results[event]["wrf"]:
            s = results[event]["wrf"][res]
            xs = mdates.date2num(s.index.to_pydatetime())
            ln, = ax.plot(xs, s.values, color="black", lw=2.5,
                          marker=("s" if SHOW_HOURLY_MARKERS else None), ms=4,
                          label="Blowing snow top (mixing ratio = 10$^{-6}$ kg kg$^{-1}$)",
                          zorder=5)
            handles.append(ln)
        # extra bs_qi iso-lines (e.g. 1e-5, 1e-4) — toggle via EXTRA_CONTOURS
        if EXTRA_CONTOURS:
            Mc = np.where(np.isfinite(M) & (M > 0), M, 0.0)
            for lev in EXTRA_CONTOURS:
                col = _extra_colors.get(lev, "#444444")
                try:
                    ax.contour(X, z_m, Mc.T, levels=[lev], colors=[col],
                               linewidths=1.3, linestyles="--", zorder=4)
                except Exception:
                    pass
                exp = int(np.round(np.log10(lev)))
                handles.append(Line2D([], [], color=col, ls="--", lw=1.3,
                                      label=f"bs_qi = 10$^{{{exp}}}$ kg kg$^{{-1}}$"))
        if handles:
            ax.legend(handles=handles, fontsize=11, loc="upper right")
        ttl = "CRYOWRF blowing snow mixing ratio" + (f" — {res}" if len(curtains) > 1 else "")
        ax.set_ylim(0, TH_YMAX); ax.set_ylabel("Height AGL (m)", fontsize=12)
        ax.tick_params(axis="both", labelsize=12)
        ax.set_title(ttl, fontsize=13, fontweight="bold")
        
    # ── hide any unused panels ──
    for j in range(n_panels, len(axl)):
        axl[j].axis("off")

    # ── time axis + xlabel on the bottom-most used panel of each column ──
    for c in range(ncols):
        col_used = [i for i in range(n_panels) if i % ncols == c]
        if not col_used:
            continue
        b = axl[max(col_used)]
        _fmt_time_axis(b)
        b.set_xlabel("Date (UTC)", fontsize=12)
        b.tick_params(labelbottom=True)                       # force ticks (sharex hides them otherwise)
        plt.setp(b.get_xticklabels(), visible=True, rotation=0)

    if cf_q is not None:
        cbar_q = fig.colorbar(cf_q, ax=q_axes, pad=0.01, shrink=0.4, aspect=30,
                              label="Blowing snow mixing ratio (kg kg⁻¹)")
        _decade_colorbar(cbar_q, TH_VMIN, max(qvmax, TH_VMIN * 10))

    plt.show()
    fig.savefig(os.path.join(OUTPUT_DIR, f"timeheight_ceilo_bsqi_{event}.png"),
                dpi=DPI, bbox_inches="tight")
    print("Saved:", f"timeheight_ceilo_bsqi_{event}.png")



In [ ]:
import math
import textwrap

In [ ]:
# ─────────────────────────────────────────────────────────────────────────
#  APPEARANCE SETTINGS — change any of these to adjust text sizes / colours
# ─────────────────────────────────────────────────────────────────────────
CEILO_CMAP        = "viridis"   # colour map for the ceilometer panel (bs_qi panels use CMAP)

BIG_TITLE_SIZE    = 16 
TITLE_SIZE        = 15          # panel titles
AXIS_LABEL_SIZE   = 15         # "Height AGL (m)", "Date (UTC)"
TICK_SIZE         = 14          # x/y tick numbers
LEGEND_SIZE       = 13          # in-panel legends
CBAR_LABEL_SIZE   = 14          # colorbar titles
CBAR_TICK_SIZE    = 14          # colorbar tick numbers
 
CBAR_WRAP         = 20          # max characters per line before a colorbar title wraps
# ─────────────────────────────────────────────────────────────────────────
 
 
def _wrap(text, width=CBAR_WRAP):
    """Wrap a colorbar label onto multiple lines if it is longer than `width`."""
    return "\n".join(textwrap.wrap(text, width=width)) if len(text) > width else text
 
 
def plot_curtains(event=None):
    event = event if event is not None else TH_EVENT
    if event == "all":
        raise ValueError("Time-height curtains are for a single event; set TH_EVENT.")
    sim = SIMULATIONS[event]
    t0, t1 = pd.Timestamp(sim["start"]), pd.Timestamp(sim["end"])
    label = results.get(event, {}).get("label", sim["label"])
 
    ceilo = load_ceilo_signal(event, t0, t1)
 
    curtains = {}
    for dom, res in DOMAIN_RES.items():
        try:
            idx, z_m, M = extract_wrf_bsqi_curtain(sim["sim_dir"], dom, t0, t1)
        except FileNotFoundError as e:
            print(f"  [{res:>5}] {dom}: MISSING — {e}"); continue
        curtains[res] = (idx, z_m, M)
    if not curtains:
        print("No model domains available for", event); return
 
    # shared bs_qi colour scale across resolutions
    qvmax = max((np.nanmax(M[np.isfinite(M) & (M > 0)]) if (np.isfinite(M) & (M > 0)).any() else TH_VMIN)
                for (_, _, M) in curtains.values())
    qlevels = _log_levels(TH_VMIN, qvmax, 50)
    qnorm   = LogNorm(vmin=TH_VMIN, vmax=max(qvmax, TH_VMIN * 10))
 
    # ── 3 rows × 2 columns grid ──
    n_panels = len(curtains) + (1 if ceilo is not None else 0)
    ncols = 2
    nrows = math.ceil(n_panels / ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(14, 2.9 * nrows),
                             squeeze=False, sharex=True, constrained_layout=True)
    axl = axes.flatten()          # row-major: [ (0,0), (0,1), (1,0), (1,1), ... ]

    ai = 0
    
    # handles for the shared figure legend
    global_handles = {}
 
    # ── ceilometer panel (first): signal − clear sky, contourf + LogNorm ──
    if ceilo is not None:
        c_idx, c_h, c_sig = ceilo
        Xc = mdates.date2num(c_idx.to_pydatetime())
        pos = c_sig[np.isfinite(c_sig) & (c_sig > 0)]
        cvmin = np.nanpercentile(pos, 5)    if pos.size else 1.0
        cvmax = np.nanpercentile(pos, 99.5) if pos.size else 10.0
        clevels = _log_levels(cvmin, cvmax, 50)
        cnorm   = LogNorm(vmin=cvmin, vmax=max(cvmax, cvmin * 10))
        Cz = np.where(np.isfinite(c_sig) & (c_sig > 0), c_sig, np.nan)  # excess only
 
        ax = axl[ai]; ai += 1
        cf = ax.contourf(Xc, c_h, Cz.T, levels=clevels, norm=cnorm,
                         cmap=CEILO_CMAP, extend="both")
        algo_top = _raw_algo_top(event, t0, t1)
        if algo_top.notna().any():
            ln_obs, = ax.plot(mdates.date2num(algo_top.index.to_pydatetime()), algo_top.values,
                              color="black", lw=2.5, label="Detected blowing snow top", zorder=5)
            global_handles["Detected blowing snow top"] = ln_obs

        ax.set_ylim(0, TH_YMAX); ax.set_ylabel("Height AGL (m)", fontsize=AXIS_LABEL_SIZE)
        ax.tick_params(axis="both", labelsize=TICK_SIZE)
        ax.set_title("Observed", fontsize=TITLE_SIZE)
        cbar_c = fig.colorbar(cf, ax=ax, pad=0.01, shrink=0.85, aspect=18)
        _decade_colorbar(cbar_c, cvmin, cvmax)
        cbar_c.set_label(_wrap("Backscatter − clear sky (sr⁻¹ m⁻¹)"), fontsize=CBAR_LABEL_SIZE)
        cbar_c.ax.tick_params(labelsize=CBAR_TICK_SIZE)
 
    # ── bs_qi panels: SAME contourf + LogNorm representation ──
    _extra_colors = {1e-5: "#ff7f0e", 1e-4: "#d62728"}
    q_axes, cf_q = [], None
    for res, (idx, z_m, M) in curtains.items():
        ax = axl[ai]; ai += 1; q_axes.append(ax)
        X  = mdates.date2num(idx.to_pydatetime())
        Mz = np.where(np.isfinite(M) & (M > 0), M, np.nan)
        cf_q = ax.contourf(X, z_m, Mz.T, levels=qlevels, norm=qnorm,
                           cmap=CMAP, extend="both")
        
        # model cloud top (height where bs_qi crosses the 1e-6 threshold)
        if event in results and res in results[event]["wrf"]:
            s = results[event]["wrf"][res]
            xs = mdates.date2num(s.index.to_pydatetime())
            ln_mod, = ax.plot(xs, s.values, color="black", lw=2.5,
                             marker=("s" if SHOW_HOURLY_MARKERS else None), ms=4,
                             label="Modelled blowing snow top", zorder=5)
            global_handles["Modelled blowing snow top"] = ln_mod

        # extra bs_qi iso-lines (e.g. 1e-5, 1e-4) — toggle via EXTRA_CONTOURS
        if EXTRA_CONTOURS:
            Mc = np.where(np.isfinite(M) & (M > 0), M, 0.0)
            for lev in EXTRA_CONTOURS:
                col = _extra_colors.get(lev, "#444444")
                try:
                    ax.contour(X, z_m, Mc.T, levels=[lev], colors=[col],
                               linewidths=1.3, linestyles="--", zorder=4)
                except Exception:
                    pass
                exp = int(np.round(np.log10(lev)))
                lbl = f"bs_qi = 10$^{{{exp}}}$ kg kg$^{{-1}}$"
                ln_extra = Line2D([], [], color=col, ls="--", lw=1.3, label=lbl)
                global_handles[lbl] = ln_extra

        ttl = "CRYOWRF " + (f" — {res} resolution" if len(curtains) > 1 else "")
        ax.set_ylim(0, TH_YMAX); ax.set_ylabel("Height AGL (m)", fontsize=AXIS_LABEL_SIZE)
        ax.tick_params(axis="both", labelsize=TICK_SIZE)
        ax.set_title(ttl, fontsize=TITLE_SIZE)
 
    # ── hide any unused panels ──
    for j in range(n_panels, len(axl)):
        axl[j].axis("off")
 
    # ── time axis + xlabel on the bottom-most used panel of each column ──
    for c in range(ncols):
        col_used = [i for i in range(n_panels) if i % ncols == c]
        if not col_used:
            continue
        b = axl[max(col_used)]
        _fmt_time_axis(b)
        b.set_xlabel("Date (UTC)", fontsize=AXIS_LABEL_SIZE)
        b.tick_params(axis="x", labelbottom=True, labelsize=TICK_SIZE)   # force ticks (sharex hides them otherwise)
        plt.setp(b.get_xticklabels(), visible=True, rotation=0)
 
    if cf_q is not None:
        cbar_q = fig.colorbar(cf_q, ax=q_axes, pad=0.01, shrink=0.4, aspect=30)
        _decade_colorbar(cbar_q, TH_VMIN, max(qvmax, TH_VMIN * 10))
        cbar_q.set_label(_wrap("Blowing snow mixing ratio (kg kg⁻¹)"), fontsize=CBAR_LABEL_SIZE)
        cbar_q.ax.tick_params(labelsize=CBAR_TICK_SIZE)

    # ── Event title ──
    EVENT_TITLES = {
        "warm_syn": "Warm Synoptic",
        "trans_syn": "Transitional Synoptic",
        "cold_kat": "Cold Katabatic"
    }
    event_title = EVENT_TITLES.get(event, label)

    fig.suptitle(f"Blowing snow cloud: observed and modelled at different resolutions \n {event_title} event", 
                 fontweight="bold", fontsize=BIG_TITLE_SIZE)

    # ── Margins and bottom legend ──
    fig.get_layout_engine().set(rect=(0, 0.07, 1, 0.94))

    if global_handles:
        fig.legend(handles=list(global_handles.values()),
                   labels=list(global_handles.keys()),
                   loc="lower center",
                   bbox_to_anchor=(0.5, 0.01),
                   ncol=len(global_handles),
                   fontsize=LEGEND_SIZE,
                   frameon=True)

    plt.show()
    fig.savefig(os.path.join(OUTPUT_DIR, f"timeheight_ceilo_bsqi_{event}.png"),
                dpi=DPI, bbox_inches="tight")
    print("Saved:", f"timeheight_ceilo_bsqi_{event}.png")

In [ ]:
plot_curtains("trans_syn")   # uses TH_EVENT; or call plot_curtains("warm_syn")


In [ ]:
plot_curtains("warm_syn") 

In [ ]:
CEILO_CMAP = "RdPu"

plot_curtains("cold_kat")   # uses TH_EVENT; or call plot_curtains("warm_syn")
